# Azure FOCUS Export — Source-Based Notebook

This version uses the package code from `src/azure_focus_export` and keeps logic in source files.

## License Notice

This notebook and project are licensed under **AGPL-3.0-only**.

If you modify this notebook or code and share it (or run modified versions for users over a network), you must make the corresponding source code available under AGPL-3.0-only.

- Full license text: `LICENSE`
- SPDX identifier: `AGPL-3.0-only`

## Prerequisites and Usage

This notebook orchestrates exports by importing code from `src/azure_focus_export`.

Before running:

1. Ensure the notebook runtime can import the package path configured in the import cell.
2. Ensure RBAC permissions are assigned:
   - `Cost Management Contributor` on the billing scope
   - `Storage Blob Data Contributor` on the storage account
3. Confirm your `config` values match the target scope and ADLS Gen2 destination.

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
%pip install -q azure-identity requests click pyyaml tenacity rich

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
import sys

# Adjust this path for your environment if needed
sys.path.insert(0, '/lakehouse/default/Files/azure-focus-export/src')

from azure_focus_export.config import AppConfig, AuthConfig, ScopeConfig, StorageConfig, ExportConfig
from azure_focus_export.auth import AzureAuthenticator
from azure_focus_export.exports_api import ExportsApiClient
from azure_focus_export.monitor import ExportMonitor
from azure_focus_export.seeder import HistoricalSeeder
from azure_focus_export.scheduler import RecurringScheduler

print('Source imports loaded.')

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
config = AppConfig(
    auth=AuthConfig(
        method='default',  # default | managed_identity | service_principal
        tenant_id=None,
        client_id=None,
        client_secret=None,
    ),
    scope=ScopeConfig(
        type='subscription',  # subscription | billing_account
        subscription_id='<your-subscription-id>',
        billing_account_id=None,
    ),
    storage=StorageConfig(
        subscription_id='<storage-subscription-id>',
        resource_group='<storage-resource-group>',
        account_name='<storage-account-name>',
        container='cost-exports',
        root_folder='focus',
    ),
    export=ExportConfig(
        history_months=36,
        export_name_prefix='focus-export',
        format='Parquet',
        compression='snappy',
    ),
)

auth = AzureAuthenticator(config.auth)
api = ExportsApiClient(auth, config)
monitor = ExportMonitor(api)
seeder = HistoricalSeeder(api, monitor, config)
scheduler = RecurringScheduler(api, config)

print('Clients initialized.')

## Execution Order

1. **Cell 1**: install dependencies
2. **Cell 2**: configure import path and load source modules
3. **Cell 3**: configure auth/scope/storage and initialize clients
4. **Cell 4**: dry-run seed preview
5. **Cell 5**: optional production seed run (uncomment)
6. **Cell 6**: dry-run recurring monthly export
7. **Cell 7**: list current export status

## Troubleshooting quick checks

- Import error: verify `sys.path.insert(...)` points to your project `src` location.
- 403 errors: verify scope/storage RBAC assignments.
- Missing exports in list: confirm `export_name_prefix` value and target scope.

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Dry-run seed preview
summary = seeder.seed(dry_run=True, skip_existing=True, batch_size=3)
summary

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Uncomment for production seed run
# summary = seeder.seed(dry_run=False, skip_existing=True, batch_size=3)
# summary

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
# Dry-run recurring monthly export
scheduler.setup_monthly_export(dry_run=True)

# Uncomment for production recurring export
# scheduler.setup_monthly_export(dry_run=False)

In [ ]:
# SPDX-License-Identifier: AGPL-3.0-only
exports = api.list_exports()
prefix = config.export.export_name_prefix
focus_exports = [e for e in exports if e.get('name', '').startswith(prefix)]

print(f'Found {len(focus_exports)} FOCUS exports')
for item in focus_exports:
    name = item.get('name', 'unknown')
    status = monitor.get_run_status(name) or 'Never run'
    print(f'- {name}: {status}')